# G1M1: Baseline vs Adaptive Window — Unified Comparator

Compare two Urc1 variants:
- **Baseline**: Fixed sliding window (standard Urc1)
- **Adaptive**: Greedy window expansion with freshness/physics gates (`Urc1_Adaptive`)

Goals:
- Use `UnifiedModelComparator` for systematic comparison across reference currents.
- Provide cross-model diagnostics (fit quality, coefficient evolution, coverage).

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_adaptive import Urc1_Adaptive
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess

In [2]:
# =============================================================================
# CONFIGURATION SECTION - EDIT THESE TO CUSTOMIZE YOUR ANALYSIS
# =============================================================================
# Dataset and directories
DATASET_PATH = r"..\\..\\explore_data\\G1M1_new.parquet"
PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"
PLOTS_OUTPUT_DIR = r"..\\plots\\adaptive\\G1M1_comparison"
SAVE_PLOTS = True

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.3,
        "Tref": 58,
        "OHref": 11,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 100,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.48,
        "Tref": 57,
        "OHref": 100,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv",
    },
}

# Shared model config
COMMON_CONFIG = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 1,
    "slide": 1,
    "min_num_data_required_for_fit": 400,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
}

# Adaptive-specific config
ADAPTIVE_CONFIG = {
    "initial_window_size": 2,
    "min_target_count_ratio": 0.1,
    "fixed_freshness_threshold": 0.2,
    "decay_rate": 0.1,
    "urc_min": 1.4,
    "urc_max": 2.4,
}

# Comparison settings
SHOW_GT_METRICS = True
SHOW_ALL_COND_METRICS = True
REF_ORDER = list(REF_CONFIGS.keys())

In [3]:
# =============================================================================
# 1. DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)

preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name

print(f"Dataset: {dataset_name}, shape: {data.shape}")
print(f"Time range: {data.index.min()} -> {data.index.max()}\n")

shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
)
print(f"Shared preprocessed rows: {len(shared_pre)}\n")

STEP 1: Data Loading & Preprocessing
=== 1. Loading & Preprocessing: G1M1_new ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_1' -> ID: '1'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\\..\\explore_data\\output\G1M1_new_20260502_123036.parquet

=== GMpreprocess Pipeline Completed Successfully ===
Dataset: G1M1_new, shape: (2529217, 3)
Time range: 2021-02-12 00:00:00 -> 2025-12-09 09:59:00

[preprocess_once] 2529217 -> 1069062 points.
Shared preprocessed rows: 1069062



In [4]:
# =============================================================================
# 2. TRAIN MODELS
# =============================================================================
print("=" * 80)
print("STEP 2: Model Training")
print("=" * 80)

models = {}

# ── 2a. Baseline ──
print("\n  -> Training Baseline (Urc1) model...")
urc_baseline = Urc1(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Baseline"] = urc_baseline
print("  OK Baseline training completed")

# ── 2b. Adaptive ──
print("\n  -> Training Adaptive (Urc1_Adaptive) model...")
urc_adaptive = Urc1_Adaptive(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
    **ADAPTIVE_CONFIG,
)
models["Adaptive"] = urc_adaptive
print("  OK Adaptive training completed")

print(f"\nOK {len(models)} models trained successfully")
print(f"\nAdaptive settings:")
for k, v in ADAPTIVE_CONFIG.items():
    print(f"  - {k}: {v}")

STEP 2: Model Training

  -> Training Baseline (Urc1) model...
Using shared preprocessed data (1069062 points, skipping preprocess).
Voltage model fitting ...
Fitting Stats: 832 intervals low data, 0 fit failed.
773 out of 1766 fitting results are reliable.
  OK Baseline training completed

  -> Training Adaptive (Urc1_Adaptive) model...
Using shared preprocessed data (1069062 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 1760 intervals...
1073 out of 1765 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=2d
  Filters: Target_Min_Ratio=0.1, Freshness=0.2
  OK Adaptive training completed

OK 2 models trained successfully

Adaptive settings:
  - initial_window_size: 2
  - min_target_count_ratio: 0.1
  - fixed_freshness_threshold: 0.2
  - decay_rate: 0.1
  - urc_min: 1.4
  - urc_max: 2.4


In [5]:
# =============================================================================
# 3. COMPARATOR SETUP & GT LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Initialize Comparator & Load Ground Truth")
print("=" * 80)

comparator = UnifiedModelComparator(models)
print(f"OK UnifiedModelComparator initialized with {len(models)} models\n")

gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_file = ref_cfg["gt_file"]
    gt_path = Path(gt_file)
    if not gt_path.is_absolute():
        gt_path = Path.cwd() / gt_path
    gt_path = gt_path.resolve()

    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break

            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(
                    f"Cannot identify voltage column: {gt_data.columns.tolist()}"
                )

            gt_series = gt_series.dropna()
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(
                f"  OK {ref_name} (Iref={iref}) GT loaded: {len(gt_series)} points [column: {selected_col}]"
            )
        except Exception as e:
            print(f"  FAILED {ref_name} GT: {e}")
    else:
        print(f"  MISSING GT file: {gt_path}")

has_gt = gt_loaded_count > 0 and SHOW_GT_METRICS
print(f"\nGT status: {gt_loaded_count}/{len(REF_CONFIGS)} references loaded")
print(f"GT metrics enabled: {has_gt}\n")


STEP 3: Initialize Comparator & Load Ground Truth
OK UnifiedModelComparator initialized with 2 models

  OK Low (Iref=0.3) GT loaded: 1762 points [column: gt_uref_regression]
  OK Medium (Iref=1.0) GT loaded: 1762 points [column: gt_uref_regression]
  OK High (Iref=1.48) GT loaded: 1384 points [column: gt_uref_regression]

GT status: 3/3 references loaded
GT metrics enabled: True



In [6]:
# =============================================================================
# 4. REFERENCE-SPECIFIC ANALYSIS (UNIFIED COMPARATOR)
# =============================================================================
print("=" * 80)
print("STEP 4: Reference-Specific Metrics & Comparison")
print("=" * 80)

rate_tables = []
for ref_name in REF_ORDER:
    ref_cfg = REF_CONFIGS[ref_name]
    iref = ref_cfg["Iref"]
    tref = ref_cfg["Tref"]
    ohref = ref_cfg["OHref"]

    print(f"\n### REFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref})")

    df_metrics = comparator.compare_all(
        i_target=iref,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )

    if df_metrics.empty:
        print("No metrics returned for this reference.")
        continue

    display(Markdown(df_metrics.to_markdown(index=False)))

    rate_tables.append(
        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(
            Reference=ref_name
        )
    )

    try:
        comparator.plot_interactive_trends(
            target_i=iref,
            show_gt=has_gt,
            save=SAVE_PLOTS,
            output_dir=PLOTS_OUTPUT_DIR,
        )
        print("OK Trend plot finished")
    except Exception as e:
        print(f"Trend plot failed: {e}")

    comparator.print_comparison_report(
        i_target=iref,
        include_gt_metrics=has_gt,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
    )

STEP 4: Reference-Specific Metrics & Comparison

### REFERENCE CONDITION: Low (Iref=0.3, Tref=58, OHref=11)


| Model Name   |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:-------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline     |                      0.3 |             11.548 |               773 |                   1.18915 |      19.388 |                    0 |               0.838 |              18 |          2.33 |             392.779 |          3.935 |               4.774 |           13.435 |                       929 | all_fitted   |         19.463 |         7.455 |                  773 |
| Adaptive     |                      0.3 |             70.147 |              1073 |                   1.23283 |       8.048 |                    0 |               0.956 |               9 |          0.84 |             195.816 |          4.116 |               4.694 |            5.35  |                      1078 | all_fitted   |          8.481 |         4.546 |                 1073 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 0.3 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      773
  Deg Rate:         1.189149 μV/h
  RMSE:             19.388 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.838
  Outliers:         18 (2.3%)
  Max Residual:     392.779 mV
  Mean SE:          3.935 mV
  Cond (Median):    10^4.8
  Cond Scope:       all_fitted (n=929)
  GT RMSE:          19.463 mV (n=773)
  GT MAE:           7.455 mV

📊 Adaptive
------------------------------------------------------------
  Data Points:      1073
  Deg Rate:         1.232825 μV/h
  RMSE:             8.048 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.956
  Outliers:         9 (0.8%)
  Max Residual:     195.816 mV
  Mean SE:          4.116 mV
  Cond (Median):    10^4.7
  Cond Scope:       all_fitted (n=1078)
  GT RMSE:          8.481 mV (n=1073)
  GT MAE:           4.546 mV

🏆 Best RMSE (vs model data):    Adaptive
🏆 Best Monotoni

| Model Name   |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:-------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline     |                        1 |             11.548 |               773 |                   5.23961 |      27.847 |                    0 |               0.955 |               9 |          1.16 |             512.973 |          3.889 |               4.774 |           13.435 |                       929 | all_fitted   |         28.94  |        16.013 |                  773 |
| Adaptive     |                        1 |             70.147 |              1073 |                   5.27305 |      17.677 |                    0 |               0.965 |              40 |          3.73 |              77     |          4.089 |               4.694 |            5.35  |                      1078 | all_fitted   |         19.275 |        14.702 |                 1073 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 1.0 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      773
  Deg Rate:         5.239611 μV/h
  RMSE:             27.847 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.955
  Outliers:         9 (1.2%)
  Max Residual:     512.973 mV
  Mean SE:          3.889 mV
  Cond (Median):    10^4.8
  Cond Scope:       all_fitted (n=929)
  GT RMSE:          28.940 mV (n=773)
  GT MAE:           16.013 mV

📊 Adaptive
------------------------------------------------------------
  Data Points:      1073
  Deg Rate:         5.273054 μV/h
  RMSE:             17.677 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.965
  Outliers:         40 (3.7%)
  Max Residual:     77.000 mV
  Mean SE:          4.089 mV
  Cond (Median):    10^4.7
  Cond Scope:       all_fitted (n=1078)
  GT RMSE:          19.275 mV (n=1073)
  GT MAE:           14.702 mV

🏆 Best RMSE (vs model data):    Adaptive
🏆 Best Monot

| Model Name   |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:-------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline     |                     1.48 |             11.548 |               773 |                   11.1444 |      53.664 |                    0 |               0.934 |              42 |          5.43 |             414.456 |          5.186 |               4.774 |           13.435 |                       929 | all_fitted   |         63.814 |        45.737 |                  591 |
| Adaptive     |                     1.48 |             70.147 |              1073 |                   10.9133 |      54.775 |                    0 |               0.914 |              61 |          5.68 |             224.78  |          5.319 |               4.694 |            5.35  |                      1078 | all_fitted   |         65.042 |        49.112 |                  844 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 1.48 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      773
  Deg Rate:         11.144427 μV/h
  RMSE:             53.664 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.934
  Outliers:         42 (5.4%)
  Max Residual:     414.456 mV
  Mean SE:          5.186 mV
  Cond (Median):    10^4.8
  Cond Scope:       all_fitted (n=929)
  GT RMSE:          63.814 mV (n=591)
  GT MAE:           45.737 mV

📊 Adaptive
------------------------------------------------------------
  Data Points:      1073
  Deg Rate:         10.913333 μV/h
  RMSE:             54.775 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.914
  Outliers:         61 (5.7%)
  Max Residual:     224.780 mV
  Mean SE:          5.319 mV
  Cond (Median):    10^4.7
  Cond Scope:       all_fitted (n=1078)
  GT RMSE:          65.042 mV (n=844)
  GT MAE:           49.112 mV

🏆 Best RMSE (vs model data):    Baseline
🏆 Best M

In [7]:
# =============================================================================
# 5. CROSS-MODEL DIAGNOSTICS + SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Cross-Model Diagnostics")
print("=" * 80)

try:
    print("\n-> Plotting fit quality (RMSE & R2 distributions)...")
    comparator.plot_fit_quality(save=SAVE_PLOTS)
    print("OK Fit quality completed")
except Exception as e:
    print(f"Fit quality plot failed: {e}")

try:
    print("\n-> Plotting coefficient diagnostics...")
    comparator.plot_coefficient_diagnostic(save=SAVE_PLOTS, include_c6=False)
    print("OK Coefficient diagnostic completed")
except Exception as e:
    print(f"Coefficient diagnostic failed: {e}")

try:
    print("\n-> Plotting coverage Gantt...")
    comparator.plot_coverage_gantt(save=SAVE_PLOTS)
    print("OK Coverage Gantt completed")
except Exception as e:
    print(f"Coverage Gantt failed: {e}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\nModels trained: {len(models)}")
print(f"Reference conditions analyzed: {len(REF_CONFIGS)}")
print(f"Ground truth loaded: {has_gt}")
print(f"Output plots dir: {PLOTS_OUTPUT_DIR}" if SAVE_PLOTS else "Plots displayed only")
print("\nAdaptive settings:")
for k, v in ADAPTIVE_CONFIG.items():
    print(f"  - {k}: {v}")

if rate_tables:
    print("\nDegradation-rate summary across references:")
    display(pd.concat(rate_tables, ignore_index=True))


STEP 5: Cross-Model Diagnostics

-> Plotting fit quality (RMSE & R2 distributions)...
OK Fit quality completed

-> Plotting coefficient diagnostics...
OK Coefficient diagnostic completed

-> Plotting coverage Gantt...
OK Coverage Gantt completed

ANALYSIS COMPLETE

Models trained: 2
Reference conditions analyzed: 3
Ground truth loaded: True
Output plots dir: ..\\plots\\adaptive\\G1M1_comparison

Adaptive settings:
  - initial_window_size: 2
  - min_target_count_ratio: 0.1
  - fixed_freshness_threshold: 0.2
  - decay_rate: 0.1
  - urc_min: 1.4
  - urc_max: 2.4

Degradation-rate summary across references:


,Model Name,Target Current (A/cm2),Degradation Rate (uV/h),Slope Sigma (uV/h),Reference
0,Baseline,0.30,1.189149,0.0,Low
1,Adaptive,0.30,1.232825,0.0,Low
2,Baseline,1.00,5.239611,0.0,Medium
3,Adaptive,1.00,5.273054,0.0,Medium
4,Baseline,1.48,11.144427,0.0,High
5,Adaptive,1.48,10.913333,0.0,High
